# Paimon Image Feature Extraction Demo

This notebook demonstrates how to ingest the COCO 2017 training split (~118k images) into Apache Paimon,
process it with PySpark and PyTorch to extract semantic features, and persist both the raw
images and the derived feature vectors back into Paimon tables.

## Overview
1. Download the [COCO 2017](https://cocodataset.org/#home) training split via the 🤗 Datasets hub (`detection-datasets/coco`) in streaming mode.
   * The train split holds 117,266 photos (within the 100k–500k target range) and the categories include people plus many animals (person, dog, cat, bird, horse, etc.).
2. Store every image as a BLOB inside a partitioned Paimon table (partitioned by the ingestion date).
   * The table definition uses Paimon's dedicated `BLOB` type together with the `blob-field` / `blob-as-descriptor` table properties and enables data evolution, so that the blob content is handled by the new blob subsystem.
3. Use PySpark to read the partition for the current day, and apply a PyTorch ResNet model to identify pictures containing humans or animals, emitting their feature maps.
4. Persist the filtered results and feature tensors in another Paimon table and visualise 10 samples with their predicted semantics.

In [ ]:
# If you run this notebook in a clean environment, install the required packages first.
!pip install --quiet datasets pyspark torch torchvision pillow matplotlib

In [ ]:
import io
import os
from datetime import date

import numpy as np
import pandas as pd
from datasets import load_dataset, get_dataset_infos
from PIL import Image
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from torchvision import models, transforms
import torch

warehouse_dir = os.path.abspath("paimon-warehouse")
os.makedirs(warehouse_dir, exist_ok=True)
ingest_date = date.today().isoformat()
print(f'Using warehouse at: {warehouse_dir}')
print(f'Ingestion date partition: {ingest_date}')

In [ ]:
# Configure Spark with the Apache Paimon catalog.
# The `spark.jars.packages` entry downloads the connector at runtime. Adjust the version if needed.
spark = (
    SparkSession.builder
    .appName("PaimonImageDemo")
    .master("local[*]")
    .config("spark.sql.extensions", "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions")
    .config("spark.sql.catalog.paimon", "org.apache.paimon.spark.SparkCatalog")
    .config("spark.sql.catalog.paimon.warehouse", warehouse_dir)
    .config("spark.jars.packages", "org.apache.paimon:paimon-spark-3-3_2.12:0.8.0")
    .getOrCreate()
)
spark.sql("SHOW DATABASES").show()

## Download the COCO 2017 dataset
We stream the train split so that the 117k images are processed incrementally rather than loaded into memory all at once.

In [ ]:
coco_info = get_dataset_infos("detection-datasets/coco")["detection-datasets--coco"]
train_examples = coco_info.splits["train"].num_examples
category_names = coco_info.features['objects'].feature['category'].names
coco_human_animal = [
    label for label in [
        "person", "bird", "cat", "dog", "horse", "sheep", "cow",
        "elephant", "bear", "zebra", "giraffe", "teddy bear"
    ]
    if label in category_names
]
print(f'Training images: {train_examples}')
print('COCO categories covering humans/animals: ' + ', '.join(coco_human_animal))
dataset = load_dataset("detection-datasets/coco", split="train", streaming=True)

## Create the raw Paimon table
The table stores the ingestion date as the partition column, records basic metadata, and keeps the original JPEG bytes within a `BLOB` field registered via the `blob-field` / `blob-as-descriptor` properties while `data-evolution` is enabled.

In [ ]:
raw_table = "paimon.default.coco_raw"
spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {raw_table} (
        image_id BIGINT,
        ingest_date STRING,
        width INT,
        height INT,
        object_count INT,
        content BLOB
    ) USING paimon
    PARTITIONED BY (ingest_date)
    TBLPROPERTIES (
        'row-tracking.enabled'='true',
        'data-evolution.enabled'='true',
        'blob-field'='content',
        'blob-as-descriptor'='true'
    )
''')
spark.sql(f"DESCRIBE FORMATTED {raw_table}").show(truncate=False)

## Ingest images into the raw table
We iterate over the streaming dataset in manageable batches. Every JPEG blob stays below 256 MB, satisfying the requirement, and we capture the number of annotated objects alongside the image metadata.

In [ ]:
batch_size = 256
buffer = []
image_counter = 0

for sample in dataset:
    image_counter += 1
    pil_image = sample['image'].convert('RGB')
    width, height = pil_image.size
    objects = sample.get('objects') or {}
    object_count = len(objects.get('category', []))
    bytes_io = io.BytesIO()
    pil_image.save(bytes_io, format='JPEG', quality=95)
    blob = bytes_io.getvalue()
    row = Row(
        image_id=int(sample['image_id']),
        ingest_date=ingest_date,
        width=int(width),
        height=int(height),
        object_count=int(object_count),
        content=bytearray(blob),
    )
    buffer.append(row)

    if len(buffer) >= batch_size:
        spark.createDataFrame(buffer).writeTo(raw_table).append()
        buffer.clear()

# flush final batch
if buffer:
    spark.createDataFrame(buffer).writeTo(raw_table).append()

print(f'Ingested {image_counter} images into {raw_table}')

## Prepare the feature extraction model
We use the torchvision ResNet-50 model pre-trained on ImageNet. The feature extractor returns the
penultimate layer embeddings, and we keep predictions for categories that represent humans or animals.

In [ ]:
weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)
model.eval()
feature_extractor = torch.nn.Sequential(*list(model.children())[:-1])
feature_extractor.eval()

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=weights.meta['mean'], std=weights.meta['std']),
])

imagenet_categories = weights.meta['categories']
target_keywords = [
    'person', 'people', 'woman', 'man', 'boy', 'girl',
    'dog', 'cat', 'bird', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'teddy',
    'animal', 'mammal', 'reptile', 'fish', 'insect'
]
target_indices = {
    idx for idx, name in enumerate(imagenet_categories)
    if any(keyword in name for keyword in target_keywords)
}
print(f'Target label count: {len(target_indices)}')

## Transform the daily partition and persist features
We rely on `mapInPandas` to combine PySpark partitioning with efficient PyTorch batch inference.

In [ ]:
feature_table = "paimon.default.coco_features"
spark.sql(f'''
    CREATE TABLE IF NOT EXISTS {feature_table} (
        image_id BIGINT,
        ingest_date STRING,
        predicted_label STRING,
        confidence DOUBLE,
        feature_vector BLOB
    ) USING paimon
    PARTITIONED BY (ingest_date)
    TBLPROPERTIES (
        'row-tracking.enabled'='true',
        'data-evolution.enabled'='true',
        'blob-field'='feature_vector',
        'blob-as-descriptor'='true'
    )
''')

source_df = (
    spark.read.table(raw_table)
    .where(F.col('ingest_date') == ingest_date)
)

def infer_partition(pdf_iter):
    for pdf in pdf_iter:
        if pdf.empty:
            continue
        images = []
        meta = []
        for row in pdf.itertuples(index=False):
            pil_image = Image.open(io.BytesIO(row.content)).convert('RGB')
            tensor = preprocess(pil_image)
            images.append(tensor)
            meta.append((row.image_id, row.ingest_date))
        batch = torch.stack(images)
        with torch.no_grad():
            logits = model(batch)
            probabilities = torch.nn.functional.softmax(logits, dim=1)
            top_prob, top_idx = torch.max(probabilities, dim=1)
            features = feature_extractor(batch).squeeze(-1).squeeze(-1)
        rows = []
        for (image_id, partition_date), pred_idx, conf, feat in zip(meta, top_idx, top_prob, features):
            idx = pred_idx.item()
            if idx not in target_indices:
                continue
            feature_bytes = feat.cpu().numpy().astype('float32').tobytes()
            rows.append((
                int(image_id),
                partition_date,
                imagenet_categories[idx],
                float(conf.item()),
                bytearray(feature_bytes),
            ))
        if rows:
            yield pd.DataFrame(rows, columns=[
                'image_id', 'ingest_date', 'predicted_label', 'confidence', 'feature_vector'
            ])

schema = (
    'image_id BIGINT, ingest_date STRING, '
    'predicted_label STRING, confidence DOUBLE, feature_vector BLOB'
)
feature_df = source_df.mapInPandas(infer_partition, schema=schema)
feature_df.writeTo(feature_table).append()
print(f'Wrote features into {feature_table}')

## Visualise ten samples
We join the feature rows with the raw images and plot a 2x5 grid showcasing the detected humans or animals.

In [ ]:
sample_features = (
    spark.read.table(feature_table)
    .where(F.col('ingest_date') == ingest_date)
    .orderBy(F.desc('confidence'))
    .limit(10)
    .toPandas()
)
sample_ids = sample_features['image_id'].tolist()
raw_samples = (
    spark.read.table(raw_table)
    .where(F.col('ingest_date') == ingest_date)
    .where(F.col('image_id').isin(sample_ids))
    .toPandas()
)
id_to_image = {
    row.image_id: Image.open(io.BytesIO(row.content)).convert('RGB')
    for row in raw_samples.itertuples(index=False)
}

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for ax, feature_row in zip(axes.flatten(), sample_features.itertuples(index=False)):
    image = id_to_image[feature_row.image_id]
    ax.imshow(image)
    ax.axis('off')
    caption = f"#{feature_row.image_id} — {feature_row.predicted_label} ({feature_row.confidence:.2f})"
    ax.set_title(caption, fontsize=10)
plt.tight_layout()
plt.show()